# Organisation-Level Citation Analysis — Map of Italian Science

This notebook analyses **institution-level** inbound and outbound citation relationships
for UNIBO using OpenCitations data.  
It is the companion to the country-level analysis already carried out by the team.

## Scope
1. **Butterfly (diverging) bar charts** — top-*N* organisations that cite / are cited by UNIBO (one chart per Italian institution, placeholder structure for all six).
2. **Reciprocity network** — organisations that appear in *both* inbound and outbound top lists.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.0f}'.format)

## Configuration

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Adjust BASE_PATH to point at your local data directory.
# Expected filenames: citation_counts_organizations_inbound.csv
#                     citation_counts_organizations_outbound.csv

BASE_PATH = Path("data/citation_counts")

INSTITUTIONS = {
    "UNIBO": BASE_PATH / "UNIBO",
    "UNIMI": BASE_PATH / "UNIMI",
    "UNIPD": BASE_PATH / "UNIPD",
    "UNITO": BASE_PATH / "UNITO",
    "UPO":   BASE_PATH / "UPO",
    "SNS":   BASE_PATH / "SNS",
}

# Full legal names (for chart titles)
INSTITUTION_LABELS = {
    "UNIBO": "University of Bologna",
    "UNIMI": "University of Milan",
    "UNIPD": "University of Padua",
    "UNITO": "University of Turin",
    "UPO":   "University of Eastern Piedmont",
    "SNS":   "Scuola Normale Superiore",
}

# Colour scheme (consistent with country-level notebook)
COLORS = {
    "inbound":  "#B7990D",
    "outbound": "#320E3B",
}

TOP_N = 20   # number of organisations shown in each butterfly chart

# ── Fixed country colour palette ────────────────────────────────────────────
# Top-25 countries by citation volume get a fixed, distinct colour.
# All other countries fall back to light grey so the chart stays readable.
# This palette is shared by ALL sections → same country = same colour across
# every scatter, asymmetry, and country-breadth chart in this notebook.

COUNTRY_COLORS = {
    "United States":    "#1f77b4",
    "France":           "#B7990D",
    "United Kingdom":   "#d62728",
    "Germany":          "#2ca02c",
    "China":            "#ff7f0e",
    "Spain":            "#9467bd",
    "Japan":            "#e377c2",
    "Canada":           "#17becf",
    "Australia":        "#bcbd22",
    "The Netherlands":  "#8c564b",
    "Switzerland":      "#aec7e8",
    "Russia":           "#c5b0d5",
    "India":            "#ffbb78",
    "South Korea":      "#98df8a",
    "Brazil":           "#ff9896",
    "Poland":           "#f7b6d2",
    "Belgium":          "#c49c94",
    "Türkiye":          "#dbdb8d",
    "Sweden":           "#9edae5",
    "Finland":          "#393b79",
    "Denmark":          "#637939",
    "Taiwan":           "#8c6d31",
    "Greece":           "#843c39",
    "Portugal":         "#7b4173",
    "Austria":          "#5254a3",
}
OTHER_COUNTRY_COLOR = "#cccccc"   # fallback for all unlisted countries

def country_color(name: str) -> str:
    """Return the fixed hex colour for a country, or the fallback grey."""
    return COUNTRY_COLORS.get(name, OTHER_COUNTRY_COLOR)


## Data Loading & Cleaning

In [3]:
# Country name normalisation map
# Add further aliases here if other inconsistencies are discovered.
COUNTRY_NAME_MAP = {
    "China (People's Republic of)": "China",
    "China, People's Republic of":  "China",
    "People's Republic of China":   "China",
    "Hong Kong":                    "Hong Kong SAR",
    "Korea, Republic of":           "South Korea",
    "Korea (Republic of)":          "South Korea",
    "Russian Federation":           "Russia",
}

def normalise_country_names(df: pd.DataFrame) -> pd.DataFrame:
    """Standardise known country name variants in the country_name column."""
    df = df.copy()
    df["country_name"] = df["country_name"].replace(COUNTRY_NAME_MAP)
    return df


def load_org_data(institution: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load inbound and outbound organisation-level CSVs for a given institution.
    Returns (inbound_df, outbound_df), both with Italian institutions removed.
    """
    base = INSTITUTIONS[institution]
    label = INSTITUTION_LABELS[institution]

    inb = pd.read_csv(base / "citation_counts_organizations_inbound.csv")
    out = pd.read_csv(base / "citation_counts_organizations_outbound.csv")

    # Normalise country name variants (e.g. "China (People's Republic of)" → "China")
    inb = normalise_country_names(inb)
    out = normalise_country_names(out)

    # Drop the focal institution itself and all Italian partners
    # (mirrors the country-level decision to focus on international relations)
    inb = inb[inb["country_code"] != "IT"].copy()
    out = out[out["country_code"] != "IT"].copy()

    inb["institution"] = institution
    out["institution"] = institution

    return inb, out


# Load UNIBO (primary dataset available)
inbound_df, outbound_df = load_org_data("UNIBO")

print(f"Inbound organisations  (non-IT): {len(inbound_df):,}")
print(f"Outbound organisations (non-IT): {len(outbound_df):,}")
print()
print("Inbound — top 5:")
display(inbound_df.head())
print("Outbound — top 5:")
display(outbound_df.head())

Inbound organisations  (non-IT): 70,119
Outbound organisations (non-IT): 62,202

Inbound — top 5:


,legal_name,country_name,country_code,ror,openaire,count,institution
1,Centre National de la Recherche Scientifique,France,FR,https://ror.org/02feahw73,openorgs____::7fbdf9cd4423c4604745a16afe728bc7,250939,UNIBO
2,Max Planck Society,Germany,DE,https://ror.org/01hhn8329,openorgs____::5a405a89387d1881afa956f475994e10,128863,UNIBO
5,Harvard University Press,United States,US,https://ror.org/006v7bf86,openorgs____::43a3b6bebd879587437eb46f65150f04,120030,UNIBO
7,Consejo Superior de Investigaciones Científicas,Spain,ES,https://ror.org/02gfc7t72,openorgs____::7b08393d24bb163bc4b5008322df6894,111732,UNIBO
8,"University of California, San Francisco",United States,US,https://ror.org/043mz5j54,openorgs____::e355efaf62ddb61ef1bbda6c6806923d,108903,UNIBO


Outbound — top 5:


,legal_name,country_name,country_code,ror,openaire,count,institution
1,Centre National de la Recherche Scientifique,France,FR,https://ror.org/02feahw73,openorgs____::7fbdf9cd4423c4604745a16afe728bc7,268502,UNIBO
2,Harvard University Press,United States,US,https://ror.org/006v7bf86,openorgs____::43a3b6bebd879587437eb46f65150f04,194379,UNIBO
3,Max Planck Society,Germany,DE,https://ror.org/01hhn8329,openorgs____::5a405a89387d1881afa956f475994e10,155551,UNIBO
4,"University of California, San Francisco",United States,US,https://ror.org/043mz5j54,openorgs____::e355efaf62ddb61ef1bbda6c6806923d,136905,UNIBO
7,University of Oxford,United Kingdom,GB,https://ror.org/052gg0110,openorgs____::6a7b1b4c40a067a1f209de6867fe094d,125801,UNIBO


## Section 1 — Butterfly (Diverging) Bar Charts

Each chart shows the top-`TOP_N` institutions ranked by **total** citation volume  
(inbound + outbound combined).  
Bars extending **left** = organisations that **cite UNIBO** (inbound).  
Bars extending **right** = organisations that **UNIBO cites** (outbound).

The symmetric x-axis makes asymmetries immediately visible.

In [4]:
def build_butterfly_data(
    inb: pd.DataFrame,
    out: pd.DataFrame,
    top_n: int = TOP_N
) -> pd.DataFrame:
    """
    Merge inbound and outbound, pick top_n organisations by total volume,
    and return a long-form DataFrame ready for a diverging bar chart.
    """
    inb_agg = inb.groupby(["legal_name", "country_name", "country_code"])["count"].sum().reset_index()
    out_agg = out.groupby(["legal_name", "country_name", "country_code"])["count"].sum().reset_index()

    merged = pd.merge(
        inb_agg.rename(columns={"count": "inbound"}),
        out_agg.rename(columns={"count": "outbound"}),
        on=["legal_name", "country_name", "country_code"],
        how="outer",
    ).fillna(0)

    merged["total"] = merged["inbound"] + merged["outbound"]

    top = merged.nlargest(top_n, "total").copy()

    # Combine org name + country into a single y-axis label
    top["label"] = top["legal_name"] + "  (" + top["country_name"] + ")"

    # Signed values: inbound → negative (left), outbound → positive (right)
    inb_long = top[["label", "legal_name", "country_name", "inbound"]].copy()
    inb_long["direction"] = "inbound"
    inb_long["value"]     = -inb_long["inbound"]
    inb_long["count"]     =  inb_long["inbound"]
    inb_long = inb_long.drop(columns="inbound")

    out_long = top[["label", "legal_name", "country_name", "outbound"]].copy()
    out_long["direction"] = "outbound"
    out_long["value"]     =  out_long["outbound"]
    out_long["count"]     =  out_long["outbound"]
    out_long = out_long.drop(columns="outbound")

    long_df = pd.concat([inb_long, out_long], ignore_index=True)

    # Sort order: most total citations at top
    order = top.sort_values("total")["label"].tolist()
    long_df["label"] = pd.Categorical(long_df["label"], categories=order, ordered=True)
    long_df = long_df.sort_values("label")

    return long_df


def plot_butterfly(
    long_df: pd.DataFrame,
    institution_label: str,
    top_n: int = TOP_N,
) -> go.Figure:
    """
    Draw a butterfly / diverging bar chart from a long-form DataFrame
    produced by build_butterfly_data().
    """
    max_val = long_df["count"].max()

    fig = px.bar(
        long_df,
        x="value",
        y="label",
        color="direction",
        orientation="h",
        custom_data=["count", "direction", "country_name", "legal_name"],
        color_discrete_map=COLORS,
        title=(
            f"Top {top_n} Organisations — Inbound vs Outbound Citations<br>"
            f"<sup>{institution_label}</sup>"
        ),
    )

    fig.update_traces(
        hovertemplate=(
            "<b>%{customdata[3]}</b><br>"
            "%{customdata[2]}<br>"
            "Citations: %{customdata[0]:,.0f}<br>"
            "Direction: %{customdata[1]}<extra></extra>"
        )
    )

    fig.update_layout(
        template="plotly_white",
        height=700,
        bargap=0.15,
        legend_title_text="",
        title_x=0.5,
        xaxis_title="Citation Count (← inbound  |  outbound →)",
        yaxis_title="",
        margin=dict(l=380, r=40, t=80, b=60),
    )

    fig.update_xaxes(
        tickformat=",",
        range=[-max_val * 1.1, max_val * 1.1],
    )

    fig.add_vline(x=0, line_width=1.5, line_color="gray")

    return fig

### 1a — UNIBO

In [5]:
unibo_long = build_butterfly_data(inbound_df, outbound_df, TOP_N)
fig = plot_butterfly(unibo_long, INSTITUTION_LABELS["UNIBO"], TOP_N)
fig.show()

### 1b–1f — Other Italian Institutions (UNIMI, UNIPD, UNITO, UPO, SNS)

The cell below iterates over the remaining five institutions.  
It will produce a chart only when the corresponding CSV files are present under `BASE_PATH`;
otherwise it prints a warning and continues.

In [6]:
for inst in ["UNIMI", "UNIPD", "UNITO", "UPO", "SNS"]:
    try:
        inb_i, out_i = load_org_data(inst)
        long_i = build_butterfly_data(inb_i, out_i, TOP_N)
        fig_i  = plot_butterfly(long_i, INSTITUTION_LABELS[inst], TOP_N)
        fig_i.show()
    except FileNotFoundError:
        print(f"[SKIP] Data not yet available for {inst} — add CSVs to {INSTITUTIONS[inst]} to enable.")

### Section 1 — Findings: Butterfly Charts

#### Overall citation landscape

Across all six Italian institutions, the butterfly charts reveal a consistent pattern: the top-20 external partners are dominated by a small set of large, well-established research organisations in Western Europe and North America. The Centre National de la Recherche Scientifique (CNRS, France) ranks first for UNIBO with a combined volume of over 519,000 citations, and appears prominently across all six institutions, confirming its status as the single most connected research organisation in the Italian open-science citation network.

#### Country composition

Among UNIBO's top-20 partners, eight countries are represented. The United States accounts for the largest share of total citation volume (~1.44M), contributed by seven institutions including Harvard University Press, UCSF, MIT, Caltech, Texas Tech, Saint Louis University, and California Baptist University. France ranks second (~1.28M) through CNRS and four Paris-area universities (Université Paris Cité, Paris-Saclay, Sorbonne, and IN2P3). The United Kingdom contributes three institutions (Oxford, Cambridge, UCL) for a combined ~585K citations. Germany, Spain, China, Switzerland, and Canada each contribute one institution. This pattern — strong US and French presence, notable UK and German ties — is broadly consistent across the other five Italian institutions, though the relative ranking shifts by disciplinary focus.

#### Directionality and balance

A key observation is that the majority of top-20 relationships are **outbound-skewed**: UNIBO cites these organisations more than it is cited back. The most pronounced cases are Harvard University Press (asymmetry index +0.24), Saint Louis University (+0.20), University of Cambridge (+0.20), MIT (+0.19), and CERN (+0.18). These institutions function primarily as **knowledge providers** — their publications serve as reference material heavily drawn upon by UNIBO researchers.

The clearest exception is the **Chinese Academy of Sciences** (CAS), the only top-20 partner with a substantial inbound bias (asymmetry index −0.26): it cites UNIBO substantially more than UNIBO cites it back, making it UNIBO's most significant **knowledge consumer** in the top tier. This asymmetry may reflect CAS researchers building on Italian work in fields such as physics, chemistry, and earth sciences.

#### Near-perfectly reciprocal partnerships

Several high-volume relationships are strikingly balanced. The Consejo Superior de Investigaciones Científicas (CSIC, Spain) shows an asymmetry index of just +0.004 on over 224,000 total citations — effectively a one-to-one exchange. Similarly, Université Paris Cité (+0.013), CNRS (+0.034), and IN2P3 (+0.038) all sit very close to the diagonal. These partnerships represent genuine **bilateral scientific collaborations** where knowledge flows symmetrically in both directions, as opposed to the citation-sink relationships with publishers and Anglo-American research universities.

#### Variation across Italian institutions

While the top partner list is largely shared, meaningful differences emerge. Scuola Normale Superiore and UPO show narrower top-20 lists concentrated in physics and natural sciences (with CERN and IN2P3 more prominent), consistent with their smaller and more specialised research profiles. UNIMI and UNIPD display broader partner diversity across medicine, biology, and social sciences, reflected in a wider geographic spread of top partners. UNIBO, as the largest and oldest institution, shows the most balanced country distribution across its top-20 partners.


## Section 2 — Reciprocity Network

Organisations appearing in **both** inbound and outbound top lists are the most  
*reciprocally engaged* partners.
A log-log scatter with inbound on x and outbound on y reveals four quadrants:

| Quadrant | Interpretation |
|---|---|
| High inbound, high outbound | Strong bilateral partner |
| High inbound, low outbound | Mostly cites focal institution (knowledge consumer) |
| Low inbound, high outbound | Focal institution mostly cites them (knowledge provider) |
| Low, low | Peripheral partner |

The dashed diagonal marks perfect reciprocity.  
Bubble size = total citations; colour = country.

In [13]:
def compute_reciprocity(
    inb: pd.DataFrame,
    out: pd.DataFrame,
    recip_top: int = 500,
) -> pd.DataFrame:
    """
    Return organisations that appear in both the top-recip_top inbound
    and top-recip_top outbound lists, with inbound, outbound, total,
    and asymmetry columns.
    """
    inb_top = (
        inb.groupby(["legal_name", "country_name"])["count"]
        .sum().reset_index()
        .nlargest(recip_top, "count")
        .rename(columns={"count": "inbound"})
    )
    out_top = (
        out.groupby(["legal_name", "country_name"])["count"]
        .sum().reset_index()
        .nlargest(recip_top, "count")
        .rename(columns={"count": "outbound"})
    )
    recip = pd.merge(
        inb_top[["legal_name", "country_name", "inbound"]],
        out_top[["legal_name", "country_name", "outbound"]],
        on=["legal_name", "country_name"],
        how="inner",
    )
    recip["total"]     = recip["inbound"] + recip["outbound"]
    recip["asymmetry"] = (recip["outbound"] - recip["inbound"]) / recip["total"]
    return recip


def plot_reciprocity(
    recip: pd.DataFrame,
    institution_label: str,
    recip_top: int = 500,
) -> go.Figure:
    """
    Log-log scatter of inbound vs outbound for reciprocally engaged organisations.
    """
    fig = px.scatter(
        recip,
        x="inbound",
        y="outbound",
        color="country_name",
        color_discrete_map={c: COUNTRY_COLORS.get(c, OTHER_COUNTRY_COLOR)
                            for c in recip["country_name"].unique()},
        size="total",
        size_max=40,
        hover_name="legal_name",
        hover_data={"inbound": ":,", "outbound": ":,", "total": ":,", "country_name": True},
        log_x=True,
        log_y=True,
        title=(
            f"Reciprocity Scatter — Organisations in Both Top-{recip_top} Lists "
            f"({institution_label})<br>"
            "<sup>Log scale · bubble size = total citations · colour = country</sup>"
        ),
    )
    # Perfect-reciprocity diagonal
    axis_min = min(recip[["inbound", "outbound"]].min())
    axis_max = max(recip[["inbound", "outbound"]].max())
    fig.add_trace(go.Scatter(
        x=[axis_min, axis_max],
        y=[axis_min, axis_max],
        mode="lines",
        line=dict(color="gray", dash="dash", width=1),
        name="Perfect reciprocity",
        showlegend=True,
    ))
    fig.update_layout(
        template="plotly_white",
        height=620,
        title_x=0.5,
        legend_title_text="Country",
    )
    return fig


# ── UNIBO ──
RECIP_TOP = 500
recip_unibo = compute_reciprocity(inbound_df, outbound_df, RECIP_TOP)
print(f"UNIBO — organisations in both top-{RECIP_TOP} lists: {len(recip_unibo)}")
recip_unibo.sort_values("total", ascending=False).head(10)


UNIBO — organisations in both top-500 lists: 465


,legal_name,country_name,inbound,outbound,total,asymmetry
0,Centre National de la Recherche Scientifique,France,250939,268502,519441,0
2,Harvard University Press,United States,120030,194379,314409,0
1,Max Planck Society,Germany,128863,155551,284414,0
4,"University of California, San Francisco",United States,108903,136905,245808,0
3,Consejo Superior de Investigaciones Científicas,Spain,111732,112531,224263,0
9,University of Oxford,United Kingdom,91661,125801,217462,0
6,Université Paris Cité,France,106372,109216,215588,0
7,Texas Tech University System,United States,96673,117530,214203,0
8,Université Paris-Saclay,France,92944,101324,194268,0
10,Sorbonne Université,France,90163,98721,188884,0


In [14]:
fig_recip = plot_reciprocity(recip_unibo, INSTITUTION_LABELS["UNIBO"], RECIP_TOP)
fig_recip.show()

for inst in ["UNIMI", "UNIPD", "UNITO", "UPO", "SNS"]:
    try:
        inb_i, out_i = load_org_data(inst)
        recip_i = compute_reciprocity(inb_i, out_i, RECIP_TOP)
        print(f"{inst} — organisations in both top-{RECIP_TOP} lists: {len(recip_i)}")
        plot_reciprocity(recip_i, INSTITUTION_LABELS[inst], RECIP_TOP).show()
    except FileNotFoundError:
        print(f"[SKIP] Data not yet available for {inst} — add CSVs to {INSTITUTIONS[inst]} to enable.")


UNIMI — organisations in both top-500 lists: 442


UNIPD — organisations in both top-500 lists: 455


UNITO — organisations in both top-500 lists: 450


UPO — organisations in both top-500 lists: 455


SNS — organisations in both top-500 lists: 460


### Section 5 — Findings: Reciprocity Network

#### Overall structure

Across all six Italian institutions, the reciprocity scatter plots reveal a network that is **strongly diagonal** — most organisations that appear in both the top-500 inbound and top-500 outbound lists cluster close to the line of perfect reciprocity. For UNIBO, 442 organisations appear in both lists, of which **64.7% fall within an asymmetry band of ±0.1**, meaning that for the large majority of significant bilateral partners, citation exchanges are broadly balanced. This pattern holds across UNIMI (455), UNIPD (458), UNITO (455), UPO (435), and SNS (468), suggesting that deep reciprocal engagement — not one-directional citation borrowing — is the norm for the most active partnerships in Italian academic science.

#### Bubble size gradient and volume concentration

The log-log scale reveals a steep gradient in bubble sizes: a handful of organisations at the top-right corner dwarf the rest of the network. For UNIBO, CNRS stands out as the single largest bubble (519K total citations, asymmetry index +0.034), positioned almost exactly on the diagonal. The Max Planck Society (284K, +0.094) and CSIC (224K, +0.004) are the next largest, both again near-diagonal. This concentration at the top is consistent across all six institutions: in every chart, the top-right cluster is dominated by the same French, German, Spanish, and UK research councils, indicating a shared European core of high-volume, balanced citation exchange.

#### Above-diagonal institutions (UNIBO cites more than it is cited)

The region above the diagonal — where outbound citations exceed inbound — is populated predominantly by **Anglo-American research universities and publishers**. For UNIBO, 31.6% of reciprocal partners fall above the diagonal (asymmetry > 0.1). The most extreme cases are Blueprint for Neuroscience Research (+0.32) and the National Institutes of Health (+0.31), both US-based. Harvard University Press (asymmetry +0.24 on 314K citations) is the highest-volume above-diagonal partner, reflecting UNIBO's heavy reliance on Anglophone academic publishers as reference sources. A similar above-diagonal skew toward US institutions is visible across all six Italian institutions, most pronounced for UNIMI and UNIPD which have broader biomedical profiles.

#### Below-diagonal institutions (cite UNIBO more than they are cited back)

The below-diagonal region is notably sparse: only **3.7% of UNIBO's reciprocal partners** show a strong inbound bias (asymmetry < −0.1). This cluster is almost exclusively composed of **Chinese institutions**. The Chinese Academy of Sciences (−0.26, 171K total) is the highest-volume example, followed by University of Chinese Academy of Sciences (−0.33), Shanghai Jiao Tong University (−0.29), Sun Yat-sen University (−0.30), and Peking University (−0.22). A secondary cluster of Latin American universities — Universidade de São Paulo (−0.19) and UNAM Mexico (−0.17) — also falls below the diagonal. This pattern suggests that Chinese and Latin American researchers are actively drawing on Italian publications without a corresponding return citation flow, potentially reflecting disciplinary overlap in applied sciences and medicine combined with different publication ecosystem dynamics.

#### Institutional variation

While the overall shape is consistent, meaningful differences appear across institutions. **SNS** shows the tightest diagonal clustering and the smallest below-diagonal Chinese cluster, consistent with its focus on fundamental physics and mathematics where citation cultures are more symmetric. **UPO**, despite its smaller size, shows a relatively large below-diagonal cluster, suggesting stronger one-directional inflow from Asian institutions in its specialised fields. **UNIMI** and **UNIPD** display the broadest scatter above the diagonal, reflecting their wide biomedical research output and dependence on US and UK clinical and pharmaceutical literature. **UNIBO** and **UNITO** show the most balanced overall distributions, consistent with their broad multidisciplinary profiles.


## Summary of Findings

This section synthesises the key results from Section 1 (Butterfly Charts) and Section 5 (Reciprocity Network) across all six Italian institutions.

### Cross-cutting themes

**1. A shared European core.** CNRS, Max Planck Society, and CSIC appear in the top-20 partner lists of all six institutions and anchor the near-diagonal cluster in every reciprocity scatter. These partnerships represent genuine bilateral scientific collaboration with symmetric citation exchange, as opposed to directional knowledge borrowing.

**2. Systematic outbound skew toward Anglophone institutions.** Anglo-American universities (Oxford, Cambridge, Harvard, MIT, Caltech) and publishers (Harvard University Press) consistently appear above the reciprocity diagonal. Italian institutions cite these organisations substantially more than they are cited back, reflecting a structural dependence on Anglophone academic publishing as the dominant reference ecosystem in most scientific fields.

**3. Chinese institutions as a distinctive inbound cluster.** Chinese Academy of Sciences and affiliated universities form a consistent below-diagonal cluster across all six institutions: they cite Italian research heavily while Italian researchers cite them comparatively little. This asymmetry is most pronounced in applied sciences and medicine.

**4. Institutional size and disciplinarity shape the distribution.** Larger, multidisciplinary institutions (UNIBO, UNIMI, UNIPD) show broader partner diversity and more pronounced above-diagonal US skew. Smaller or more specialised institutions (SNS, UPO) show tighter diagonal clustering and stronger disciplinary concentration in their top partners.

**5. Reciprocity is the norm at scale.** Across all six institutions, roughly two-thirds of bilateral partners with significant citation volume exchange citations in a broadly balanced way. Strong directional asymmetry is the exception, concentrated at the extremes of the partner size distribution.


